In [2]:
from chromadb import PersistentClient
client = PersistentClient(path="chroma_store")
collection = client.get_or_create_collection(name="wiki_chunks")


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Step 1: Define query
query = "Who is marieve herington?"

# Step 2: Embed query
embedder = SentenceTransformer("all-MiniLM-L6-v2")
query_embedding = embedder.encode(query).tolist()

# Step 3: Retrieve top 5 chunks from ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5  # number of chunks you want
)

# Step 4: Print the top 5 chunks
retrieved_chunks = results['documents'][0]  # A list of top 5 matched text chunks
print("Top 5 chunks:")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}\n")


Top 5 chunks:
1. com e marieve herington facebook marieve herington on twitter e marieve herington on instagram categories 1988 births living people actresses from los angeles actresses from oakville, ontario canadian child actresses canadian emigrants to the united states canadian film actresses canadian jazz singers canadian television actresses canadian video game actresses canadian voice actresses canadian women jazz singers franco ontarian people jazz musicians from california singers from los angeles university of toronto alumni 21st century canadian actresses 21st century canadian women singers

2. and members of the royal jelly orchestra, appearing on two more albums. she sang the theme songs for cbc television s sesame park, nelvana s pippi longstocking and tvontario s marigold s mathemagics. herington moved to los angeles in 2008. 2i the marieve herington band performs in southern california and toronto. their latest album is midnight sessions. on april 14, 2021, herington an

In [13]:
from transformers import pipeline

# Load a QA pipeline with a pretrained model fine-tuned on SQuAD
qa_pipeline = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad")

query = "Who is marieve herington?"

# Assuming retrieved_chunks is a list of context chunks (text passages)
retrieved_chunks = [
    "Marieve Herington is a Canadian actress and singer...",
    "She is known for her roles in animation and television...",
    # add more relevant text chunks here
]

answers = []

for chunk in retrieved_chunks:
    if chunk.strip():  # skip empty chunks
        result = qa_pipeline(question=query, context=chunk)
        answers.append(result['answer'])

# Combine answers or pick the best one
# Simple approach: pick the longest answer (you can use other heuristics)
final_answer = max(answers, key=len) if answers else "No answer found."

print("Final Answer:\n", final_answer)


config.json:   0%|          | 0.00/451 [00:00<?, ?B/s]

C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--distilbert-base-uncased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling bac

model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


Final Answer:
 She is known for her roles in animation and television


In [25]:
query = "Who is marieve herington?"

In [26]:
def vector_search(query, top_k=5):
    # This depends on how your vector DB is set up.
    # Here we query the collection by similarity to query text.
    results = collection.query(query_texts=[query], n_results=top_k)
    # results['documents'] is a list of lists of chunks (strings)
    return results['documents'][0] if results['documents'] else []

retrieved_chunks = vector_search(query)


In [45]:
from transformers import pipeline

llm = pipeline("text-generation", model="distilgpt2")

# Map step
mapped_answers = [llm(f"{chunk}\n\nQuestion: {query}", max_new_tokens=100)[0]['generated_text'] for chunk in retrieved_chunks]

# Reduce step
reduce_prompt = "Think step by step and explain your answer.\n\n" + "\n\n".join(mapped_answers)
final_answer = llm(reduce_prompt, max_new_tokens=200)[0]['generated_text']

print("Final Answer:\n", final_answer)


Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Final Answer:
 Think step by step and explain your answer.

Marieve Herington is a Canadian actress, voice actress, and singer born in 1988. She has worked in film, television, and video games. She is from Oakville, Ontario.

Question: Who is Marieve Herington?
Marieve Herington is a Canadian actress, voice actress, and singer born in 1988. She has worked in film, television, and video games. She is from Oakville, Ontario.
The name Marieve Herington means "she" on both the shortlist and the shortlist.
Marieve Herington is known online for the role of Dr. M. T. Robinson in American Horror Story; and recently was an American TV reality show writer for the upcoming season

She is known for her jazz singing and has worked in both Canadian and U.S. entertainment industries. University of Toronto alumna.

Question: Who is Marieve Herington?
Marieve Herington is a writer, journalist and activist on behalf of the voiceless against injustice or oppression. Marieve Herington is a writer, journal

In [51]:
def rag_chat(query, chat_history):
    print(f"\n Received query: {query}")

    query_embedding = embedder.encode(query).tolist()
    print(" Query embedded.")

    retrieved = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=['documents']
    )
    retrieved_chunks = retrieved['documents'][0]
    print(f" Retrieved {len(retrieved_chunks)} chunks.")

    mapped_answers = []
    for i, chunk in enumerate(retrieved_chunks):
        print(f"\n Processing chunk {i+1}:\n{chunk[:150]}...")
        try:
            response = llm(f"{chunk}\n\nQuestion: {query}", max_new_tokens=200)
            answer = response[0]['generated_text']
            mapped_answers.append(answer)
            print(f"Answer {i+1} received.")
        except Exception as e:
            print(f" Error processing chunk {i+1}: {e}")
            mapped_answers.append("")

    final_prompt = "Summarize the following answers. Think step by step and explain your answer:\n\n" + "\n".join(mapped_answers)
    print("\n Reducing answers...")

    try:
        response = llm(final_prompt, max_new_tokens=300)
        final_answer = response[0]['generated_text']
        print("\n Final summarized answer:")
        print(final_answer)
    except Exception as e:
        print(f" Error during reduction: {e}")
        final_answer = "There was an error generating the final answer."

    chat_history.append({"user": query, "bot": final_answer})
    return final_answer, chat_history


In [52]:
response, chat_history = rag_chat("Who is Marieve Herington?", chat_history)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 Received query: Who is Marieve Herington?
 Query embedded.
 Retrieved 5 chunks.

 Processing chunk 1:
com e marieve herington facebook marieve herington on twitter e marieve herington on instagram categories 1988 births living people actresses from los...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer 1 received.

 Processing chunk 2:
and members of the royal jelly orchestra, appearing on two more albums. she sang the theme songs for cbc television s sesame park, nelvana s pippi lon...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

Answer 3 received.

 Processing chunk 4:
of olivia and john s daughters and third child, born in waltons series chronology in april 1920, then aged 13 in season one. throughout the first few ...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer 4 received.

 Processing chunk 5:
2il 9 she has additional breton heritage. beyoncé s fourth great grandmother, marie frangoise trahan, was born in 1774 in bangor, located on belle jle...


Token indices sequence length is longer than the specified maximum sequence length for this model (2476 > 1024). Running this sequence through the model will result in indexing errors
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer 5 received.

 Reducing answers...
 Error during reduction: index out of range in self
